In [2]:
test = 1,2,3
print(test)
print(type(test))


(1, 2, 3)
<class 'tuple'>


In [16]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
customers = pd.DataFrame({
    'customer_id': np.arange(1,201),
    'region': rng.choice(['서울', '경기', '부산', '광주'], 200),
    'joined': pd.to_datetime('2025-01-01') + + pd.to_timedelta(rng.integers(0, 365, 200))
})

customers.to_csv('customers.csv', index=False, encoding='utf-8')

In [17]:
df = pd.read_parquet('orders_step1.parquet')
print('orders:', df.shape, '/ customers:', customers.shape)

orders: (1015, 10) / customers: (200, 3)


In [ ]:
import pandas as pd
 
df = pd.read_parquet('orders_step1.parquet')
customers = pd.read_csv('customers.csv', parse_dates=['joined'])

# # 2단계 — 카테고리별 명명 집계
# summary = df.groupby('category').agg(
#     orders=('order_id', 'count'),
#     total=('amount', 'sum'),
#     avg_price=('price', 'mean'),
# ).round(0).sort_values('total', ascending=False)
# print(summary)
# print('열 이름:', summary.columns.tolist())

          orders        total  avg_price
category                                
식품           416  116106000.0   100072.0
의류           284   81607000.0    98359.0
가전           203   59060000.0   101891.0
도서           112   31213000.0   103565.0
열 이름: ['orders', 'total', 'avg_price']


In [31]:
summary = df.groupby('category').agg(
    orders=('order_id','count'),
    total=('amount','sum'),
    avg_price=('price', 'mean')
).round(0).sort_values('total', ascending=False)

print(summary)
print('열 이름:', summary.columns.to_list())

          orders        total  avg_price
category                                
식품           416  116106000.0   100072.0
의류           284   81607000.0    98359.0
가전           203   59060000.0   101891.0
도서           112   31213000.0   103565.0
열 이름: ['orders', 'total', 'avg_price']


In [34]:
multi = df.groupby('category').agg({'amount': ['sum','mean']})

In [36]:
multi.columns.to_list()
multi.head(2)

amount               
                 sum           mean
category                           
가전        59060000.0  293830.845771
도서        31213000.0  289009.259259

In [41]:
by2= df.groupby(['category','month']).agg(
    orders=('order_id', 'count'),
    total=('amount', 'sum')
)
by2.head(6)

flat = by2.reset_index()
flat.head(3)
len(flat)


8

In [45]:
m = df.merge(customers, on='customer_id', how='left', validate='many_to_one')

print('결합 전:', len(df), '행 / 결합 후:', len(m), '행')
print('region 결측:', m['region'].isna().sum(), '건')

결합 전: 1015 행 / 결합 후: 1015 행
region 결측: 0 건


In [47]:
bad = pd.concat([customers, customers.head(5)], ignore_index=True)
print('customers 원본:', len(customers), '/ 중복 심은 것:', len(bad))

customers 원본: 200 / 중복 심은 것: 205


In [48]:
m_bad = df.merge(bad, on='customer_id', how='left')
print('결합 후 행 수:', len(m_bad), '(원본', len(df), ')')

결합 후 행 수: 1040 (원본 1015 )


In [49]:
print('결합 후 행 수:', len(m_bad), '(원본', len(df), ')')
print('매출 합계 원본:', int(df['amount'].sum()))
print('매출 합계 결합후:', int(m_bad['amount'].sum()), '← 부풀려졌습니다')

결합 후 행 수: 1040 (원본 1015 )
매출 합계 원본: 287986000
매출 합계 결합후: 297258000 ← 부풀려졌습니다


In [50]:
try:
    df.merge(bad, on='customer_id', how='left', validate='many_to_one')
except Exception as e:
    print('예상대로 차단:', type(e).__name__)
    print(e)

예상대로 차단: MergeError
Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  customer_id
           1
           2
           3
           4
           5 ...


In [51]:
# 5단계(습관) — analyze.py 끝에 이어 붙입니다
# 결합 전에 키 중복을 미리 확인하는 습관
print('오른쪽 키 중복 건수:', bad['customer_id'].duplicated().sum())
print('정상 마스터 중복 건수:', customers['customer_id'].duplicated().sum())

오른쪽 키 중복 건수: 5
정상 마스터 중복 건수: 0


In [54]:
pv = m.pivot_table(index='region', columns='category',
                   values='amount', aggfunc='sum', fill_value=0)
print(pv)
print('교차표 총합 == 전체 매출:', pv.to_numpy().sum() == m['amount'].sum())


category          가전          도서          식품          의류
region                                                  
경기        17543000.0   7965000.0  29052000.0  18363000.0
광주        13177000.0   6755000.0  28030000.0  20705000.0
부산        13858000.0  11401000.0  31449000.0  20681000.0
서울        14482000.0   5092000.0  27575000.0  21858000.0
교차표 총합 == 전체 매출: True


In [57]:
print(m.pivot_table(index='region', columns='category',values='order_id', aggfunc='count', fill_value=0))
print(m.pivot_table(index='region', columns='category',
                    values='amount', aggfunc='mean', fill_value=0).round(0))


category  가전  도서   식품  의류
region                   
경기        56  28  104  66
광주        46  29   98  77
부산        49  36  117  74
서울        52  19   97  67
category        가전        도서        식품        의류
region                                          
경기        313268.0  284464.0  290520.0  282508.0
광주        286457.0  241250.0  301398.0  279797.0
부산        294851.0  345485.0  271112.0  283301.0
서울        278500.0  268000.0  303022.0  341531.0


In [63]:
long = pv.reset_index().melt(id_vars='region',
                             var_name='category', value_name='amount')

print(long.head())
print('행 수:', len(long), '(지역 4 × 카테고리 4)')
 
# 긴 형태 → 다시 넓게
wide = long.pivot(index='region', columns='category', values='amount')
print('원래대로 돌아왔는가:', wide.equals(pv))

  region category      amount
0     경기       가전  17543000.0
1     광주       가전  13177000.0
2     부산       가전  13858000.0
3     서울       가전  14482000.0
4     경기       도서   7965000.0
행 수: 16 (지역 4 × 카테고리 4)
원래대로 돌아왔는가: True


In [64]:
# timeseries.py — 1~6단계를 이 파일에 이어 붙입니다
import pandas as pd

df = pd.read_parquet('orders_step1.parquet')
customers = pd.read_csv('customers.csv', parse_dates=['joined'])
m = df.merge(customers, on='customer_id', how='left', validate='many_to_one')

# 일부러 실패 — 인덱스가 날짜가 아닌 채로 resample
try:
    m['amount'].resample('D').sum()
except TypeError as e:
    print('예상대로 실패:', e)

예상대로 실패: Only valid with DatetimeIndex, TimedeltaIndex or PeriodIndex, but got an instance of 'RangeIndex'


In [71]:
# 1단계 — timeseries.py 끝에 이어 붙입니다
ts = m.set_index('order_date').sort_index()
print('인덱스 타입:', ts.index.dtype)
print('기간:', ts.index.min().date(), '~', ts.index.max().date())
print(ts.head(2))

인덱스 타입: datetime64[us]
기간: 2026-06-01 ~ 2026-07-30
            order_id  customer_id category     price  quantity    amount  \
order_date                                                                 
2026-06-01       719           48       의류  136000.0         3  408000.0   
2026-06-01       438          186       식품  132000.0         5  660000.0   

            year  month weekday region                        joined  
order_date                                                            
2026-06-01  2026      6  Monday     서울 2025-01-01 00:00:00.000000187  
2026-06-01  2026      6  Monday     경기 2025-01-01 00:00:00.000000014  


In [83]:
daily = ts['amount'].resample('D').sum()
weekly_trend = daily.rolling(7).mean()

print('일별 매출 (마지막 3일)')
print(daily.tail(3))
print('7일 이동 평균 (마지막 3일)')
print(weekly_trend.round(0).tail(3))
 
print('daily 행 수:', len(daily))
print('rolling 행 수:', len(weekly_trend), '← 같습니다')


일별 매출 (마지막 3일)
order_date
2026-07-28    2420000.0
2026-07-29    4323000.0
2026-07-30    4587000.0
Freq: D, Name: amount, dtype: float64
7일 이동 평균 (마지막 3일)
order_date
2026-07-28    4734571.0
2026-07-29    4834571.0
2026-07-30    4562714.0
Freq: D, Name: amount, dtype: float64
daily 행 수: 60
rolling 행 수: 60 ← 같습니다


In [84]:
print(weekly_trend.head(8))


order_date
2026-06-01             NaN
2026-06-02             NaN
2026-06-03             NaN
2026-06-04             NaN
2026-06-05             NaN
2026-06-06             NaN
2026-06-07    4.040000e+06
2026-06-08    4.566143e+06
Freq: D, Name: amount, dtype: float64


In [86]:
weekly = ts['amount'].resample('W').sum()
monthly = ts['amount'].resample('ME').agg(['sum', 'mean', 'count'])

print('주별 매출')
print(weekly)
print('\n월별 요약')
print(monthly.round(0))

주별 매출
order_date
2026-06-07    28280000.0
2026-06-14    37291000.0
2026-06-21    32125000.0
2026-06-28    33634000.0
2026-07-05    37971000.0
2026-07-12    33278000.0
2026-07-19    33887000.0
2026-07-26    36611000.0
2026-08-02    14909000.0
Freq: W-SUN, Name: amount, dtype: float64

월별 요약
                    sum      mean  count
order_date                              
2026-06-30  141446000.0  297155.0    476
2026-07-31  146540000.0  287898.0    509


In [87]:
print(m[['price', 'quantity', 'amount']].describe().round(0))

          price  quantity    amount
count     985.0    1015.0     985.0
mean   100346.0       3.0  292372.0
std     56796.0       1.0  228761.0
min      1000.0       1.0    2000.0
25%     49000.0       2.0  114000.0
50%    101000.0       3.0  222000.0
75%    150000.0       4.0  432000.0
max    199000.0       5.0  995000.0


In [88]:
print('\n주문 금액 분위수')
print(m['amount'].quantile([0.25, 0.5, 0.75, 0.9, 0.99]).round(0))
 


주문 금액 분위수
0.25    114000.0
0.50    222000.0
0.75    432000.0
0.90    631200.0
0.99    950800.0
Name: amount, dtype: float64


In [89]:
print('\n지역별')
print(m.groupby('region')['amount'].agg(['mean', 'std', 'count']).round(0))


지역별
            mean       std  count
region                           
경기      292863.0  226617.0    249
광주      284925.0  223572.0    241
부산      287691.0  223812.0    269
서울      305341.0  242944.0    226


In [90]:
m.to_parquet('orders_enriched.parquet', index=False)
print('저장 완료:', len(m), '행,', len(m.columns), '열')
print('열 목록:', m.columns.tolist())

저장 완료: 1015 행, 12 열
열 목록: ['order_id', 'customer_id', 'category', 'price', 'quantity', 'order_date', 'amount', 'year', 'month', 'weekday', 'region', 'joined']


In [91]:
# 6단계(미니 과제 정답 예시) — timeseries.py 끝에 이어 붙입니다
cust = m.groupby('customer_id').agg(
    orders=('order_id', 'count'),
    total=('amount', 'sum'),
    last_order=('order_date', 'max'),
).sort_values('total', ascending=False)
print(cust.head(10))

print(m.groupby('weekday')['amount'].mean().round(0).sort_values(ascending=False))

top = (m.groupby(['region', 'category'])['amount'].sum()
         .reset_index()
         .sort_values('amount', ascending=False)
         .groupby('region').head(1))
print(top)

             orders      total last_order
customer_id                              
19               12  4452000.0 2026-07-25
90                9  3706000.0 2026-07-26
88                8  3392000.0 2026-07-30
170               8  3284000.0 2026-07-29
105               8  3231000.0 2026-07-27
2                 8  3226000.0 2026-07-23
194               8  3132000.0 2026-07-23
119               6  3111000.0 2026-07-17
9                 9  3065000.0 2026-07-27
157              11  3031000.0 2026-07-26
weekday
Wednesday    306961.0
Sunday       305180.0
Saturday     301304.0
Monday       290086.0
Friday       289357.0
Tuesday      285400.0
Thursday     274162.0
Name: amount, dtype: float64
   region category      amount
10     부산       식품  31449000.0
2      경기       식품  29052000.0
6      광주       식품  28030000.0
14     서울       식품  27575000.0
